# Парсер DDL-ошибок

## Импорты

In [1]:
import csv
import re
from pathlib import Path
from collections import defaultdict
from datetime import datetime

## Пути к файлам

In [2]:
INPUT_CSV = Path(r"C:\Users\msmer\Desktop\parser_error_air\errors (1).csv")
OUT_CSV = Path(r"C:\Users\msmer\Desktop\parser_error_air\errors_report.csv")

## Справочник категорий ошибок

In [3]:
ERROR_DICT = {
    "partition_no_specified": "Не сгенерированы значения для LIST-партиций",
    "invalid_partition_boundary": "Невалидная граница для LIST-партиции (смешан RANGE/LIST)",
    "relation_already_exists": "Длина имени партиции превышает 63 символа",
    "missing_column": "Отсутствует поле в зависимом объекте",
    "missing_relation": "Отсутствует зависимый объект",
    "syntax_error": "Синтаксическая ошибка в SQL-скрипте",
    "task_group_not_registered": "Код группы задач отсутствует в sys_task_group",
    "object_attr_not_registered": "Нет атрибутов таблицы в system_core.sys_object_attrs",
    "attribute_not_registered": "Нет атрибута в system_core.sys_object_attrs",
    "other": "Неизвестная ошибка"
}

## Классификация ошибок

In [4]:
def nq(s: str) -> str:
    return s.replace('""', '"')


def classify(err: str) -> str:
    e = nq(err)

    if "no partitions specified at depth" in e:
        return "partition_no_specified"

    if "invalid boundary specification for LIST partition" in e:
        return "invalid_partition_boundary"

    if "already exists" in e:
        return "relation_already_exists"

    if "column " in e and "does not exist" in e:
        return "missing_column"

    if ("relation " in e or "schema " in e) and "does not exist" in e:
        return "missing_relation"

    if "syntax error" in e:
        return "syntax_error"

    if "task_group_code" in e:
        return "task_group_not_registered"

    if "sys_object_attrs" in e:
        return "object_attr_not_registered"

    if "attribute_name" in e:
        return "attribute_not_registered"

    return "other"

## Извлечение зависимостей

In [5]:
def extract_dependency(err: str, cat: str) -> str:
    e = nq(err)

    if "prt" in e.lower() and "already exists" in e.lower():
        m = re.search(r'relation\s+"([^"]+)"', e)
        if m:
            return m.group(1)

    if cat in ("missing_relation", "relation_already_exists"):
        m = re.search(r'(relation|schema)\s+"([^"]+)"', e)
        return m.group(2) if m else ""

    if cat == "missing_column":
        m = re.search(r"column\s+([^\s]+)\s+does not exist", e)
        return m.group(1) if m else ""

    if cat == "partition_no_specified":
        m = re.search(r'partition by list\s*\(\s*"([^"]+)"', e)
        return m.group(1) if m else ""

    if cat == "invalid_partition_boundary":
        m = re.search(r"subpartition\s+(\S+)", e)
        return m.group(1) if m else ""

    if cat == "task_group_not_registered":
        m = re.search(r"task_group_code\s+\[(.*?)\]", e)
        return m.group(1).strip() if m else ""

    if cat == "object_attr_not_registered":
        m = re.search(
            r"схемы \[(.*?)\], объекта \[(.*?)\] и атрибута \[(.*?)\]",
            e
        )
        return ".".join(m.groups()) if m else ""

    if cat == "attribute_not_registered":
        m = re.search(r"attribute_name \((.*?)\)", e)
        return m.group(1) if m else ""

    if cat == "syntax_error":
        m = re.search(r'at or near\s+""?([^"\s]+)""?', e)
        return m.group(1) if m else ""

    return ""

## Проверка длины имени партиции

In [6]:
def is_partition_length_error(dep: str, err: str) -> bool:
    e = nq(err).lower()

    if "prt" in dep.lower() or "prt" in e:
        if (
            "already exists" in e
            or "syntax error" in e
            or len(dep) > 63
        ):
            return True

    return False

## Формирование описания проблемы

In [7]:
def get_problem_string(cat: str, dep: str, err: str) -> str:
    dep_str = f'"{dep}"' if dep else ""

    if cat == "relation_already_exists":
        if "prt" in err.lower():
            return f"Длина имени партиции > 63 символов: {dep_str}"

        return f"Объект уже существует: {dep_str}"

    if cat == "missing_relation":
        if dep and "." in dep:
            return f"Не найдена зависимая таблица/вьюха в источнике: {dep_str}"

        return f"Не найден зависимый CTE: {dep_str}"

    if cat == "missing_column":
        return f"В зависимом объекте отсутствует поле: {dep_str}"

    if cat == "task_group_not_registered":
        if dep:
            return (
                f"Код группы задач (task_group_code) "
                f"отсутствует в sys_task_group: {dep_str}"
            )

        return "В настройках задачи не указан код группы (task_group_code пустой)"

    if cat == "partition_no_specified":
        return f"Не сгенерированы значения для LIST-партиций по ключу: {dep_str}"

    if cat == "invalid_partition_boundary":
        return f"Невалидная граница для LIST-партиции: {dep_str}"

    if cat == "syntax_error":
        return f"Синтаксическая ошибка в скрипте около: {dep_str}"

    if cat == "object_attr_not_registered":
        return (
            f"Атрибуты не найдены в sys_object_attrs "
            f"(Схема.Объект.Поле): {dep_str}"
        )

    if cat == "attribute_not_registered":
        return f"Атрибут не найден в sys_object_attrs: {dep_str}"

    return ERROR_DICT.get(cat, "Неизвестная ошибка")

## Определение объекта по пути

In [8]:
def path_obj(path: str) -> str:
    p = path.replace("\\", "/")

    m = re.search(r"/out_with_array/([^/]+)/([^/]+)/", p)

    if m:
        return f"{m.group(1)}.{m.group(2)}"

    return Path(path).name

## Чтение и обработка исходного CSV

In [9]:
def parse(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Не найден входной файл: {path}")

    text = path.read_text(encoding="utf-8-sig")

    records = []
    seen = set()
    row_num = 0

    for raw in text.splitlines():
        line = raw.strip()

        if (
            not line
            or line.startswith("file_path;error")
            or line.startswith("---")
        ):
            continue

        inactive = False

        if line.startswith("НЕАКТУАЛЬНО "):
            inactive = True
            line = line[len("НЕАКТУАЛЬНО "):].strip()

        if not re.match(r"^[A-Za-z]:\\", line) or ";" not in line:
            continue

        file_path, err = line.split(";", 1)

        file_path = file_path.strip()
        err = err.strip()

        if not file_path or not err:
            continue

        row_num += 1

        cat = classify(err)
        dep = extract_dependency(err, cat)

        obj = path_obj(file_path)
        problem_text = get_problem_string(cat, dep, err)

        status = "уникальная"

        if inactive:
            status = "неактуальная"
        elif (obj, err) in seen:
            status = "дубликат"

        seen.add((obj, err))

        was_value = ""

        if is_partition_length_error(dep, err):
            was_value = dep

        elif cat in (
            "missing_column",
            "missing_relation",
            "syntax_error"
        ):
            was_value = dep

        records.append({
            "num": row_num,
            "status": status,
            "obj": obj,
            "cat": cat,
            "problem": problem_text,
            "err": err,
            "dep": dep,
            "was": was_value
        })

    return records

## Вывод статистики

In [10]:
def print_statistics(records):
    total = len(records)

    stats_cat = defaultdict(int)
    stats_status = defaultdict(int)
    stats_obj = defaultdict(int)

    for r in records:
        stats_cat[r["cat"]] += 1
        stats_status[r["status"]] += 1
        stats_obj[r["obj"]] += 1

    unique_objects = len(stats_obj)

    print("\n" + "=" * 75)
    print("                  СТАТИСТИКА ОБРАБОТКИ DDL-ОШИБОК")
    print("=" * 75)

    print(f"Дата обработки          : {datetime.now():%d.%m.%Y %H:%M:%S}")
    print(f"Всего найдено ошибок    : {total}")
    print(f"Количество объектов     : {unique_objects}")

    print("\n" + "-" * 75)
    print("СТАТУСЫ ОШИБОК")
    print("-" * 75)

    for status in ("уникальная", "дубликат", "неактуальная"):
        count = stats_status.get(status, 0)
        percent = count / total * 100 if total else 0

        print(
            f"{status.capitalize():20} "
            f"{count:8} шт. ({percent:6.2f}%)"
        )

    print("\n" + "-" * 75)
    print("РАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ")
    print("-" * 75)

    for cat, count in sorted(
        stats_cat.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        percent = count / total * 100 if total else 0
        name = ERROR_DICT.get(cat, cat)

        print(
            f"{count:6} шт. ({percent:6.2f}%) | {name}"
        )

    print("\n" + "-" * 75)
    print("ТОП-10 ОБЪЕКТОВ ПО КОЛИЧЕСТВУ ОШИБОК")
    print("-" * 75)

    for obj, count in sorted(
        stats_obj.items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]:
        print(f"{count:6} шт. | {obj}")

    print("=" * 75 + "\n")

## Создание итогового CSV

In [11]:
def save_report(records, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(
        output_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as f:
        writer = csv.writer(
            f,
            delimiter=";",
            quoting=csv.QUOTE_MINIMAL
        )

        writer.writerow([
            "№",
            "Статус ошибок",
            "Объект",
            "Проблема",
            "Исходный код ошибки",
            "Источник на ODS",
            "Тип данных на ODS",
            "Поле",
            "Было",
            "Стало"
        ])

        for r in records:
            writer.writerow([
                r["num"],
                r["status"],
                r["obj"],
                r["problem"],
                r["err"],
                "",
                "",
                "",
                r["was"],
                ""
            ])

## Запуск обработки

In [12]:
def run_parser():
    print(f"Исходный файл: {INPUT_CSV}")

    records = parse(INPUT_CSV)

    print_statistics(records)
    save_report(records, OUT_CSV)

    print("ОБРАБОТКА ЗАВЕРШЕНА")
    print("-" * 75)
    print(f"Итоговый файл: {OUT_CSV.resolve()}")
    print(f"Записей в отчёте: {len(records)}")

## Выполнение

In [13]:
run_parser()

Исходный файл: C:\Users\msmer\Desktop\parser_error_air\errors (1).csv

                  СТАТИСТИКА ОБРАБОТКИ DDL-ОШИБОК
Дата обработки          : 09.11.2026 18:03:51
Всего найдено ошибок    : 274
Количество объектов     : 123

---------------------------------------------------------------------------
СТАТУСЫ ОШИБОК
---------------------------------------------------------------------------
Уникальная                177 шт. ( 64.60%)
Дубликат                   75 шт. ( 27.37%)
Неактуальная               22 шт. (  8.03%)

---------------------------------------------------------------------------
РАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ
---------------------------------------------------------------------------
   163 шт. ( 59.49%) | Отсутствует зависимый объект
    28 шт. ( 10.22%) | Код группы задач отсутствует в sys_task_group
    25 шт. (  9.12%) | Длина имени партиции превышает 63 символа
    25 шт. (  9.12%) | Отсутствует поле в зависимом объекте
    15 шт. (  5.47%) | Нет атрибутов таблицы в